# How does an AI read *your* documents? A hands-on tour of RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hugobb/IndigenousAI/blob/main/notebooks/rag_workshop_beginner.ipynb)

Chatbots answer questions from **memory**: everything they learned while being trained. They have never seen *your* documents, like a community report, meeting notes or an archive. **RAG** (*Retrieval-Augmented Generation*) is the most common way to fix this. Before answering, the system **looks up** the most relevant passages in documents you choose, and then the AI **writes its answer from those passages**.

In this workshop you will take a RAG system apart, piece by piece, and play with each piece to see what it does.

**No coding needed.** You never have to type or read code. Each grey box with a ▶️ button is something to run: click the ▶️ on its left (or click inside it and press **Shift + Enter**). Then use the sliders, buttons and text boxes that appear. If you are curious about the code behind a box, double-click its title to show it.

| Part | What it's about | Time |
|---|---|---|
| 0 | Setup | ⏱ 10 min |
| 1 | The problem: an AI that guesses | ⏱ 5 min |
| 2 | Embeddings: turning meaning into numbers | ⏱ 20 min |
| 3 | Search: finding the right passage | ⏱ 10 min |
| 4 | Chunking: cutting documents into pieces | ⏱ 20 min |
| 5 | Generation: writing the answer | ⏱ 15 min |
| 6 | Take it home | ⏱ 10 min |
| ★ | Bonus: try your own documents | at home |

## 0. Setup ⏱ about 10 min

1. **Check you have a GPU** (a fast chip for AI). This notebook asks Colab for one automatically. To check, open the menu **Runtime → Change runtime type**, make sure **T4 GPU** is selected, and click **Save**.
2. **Run the Setup box below** by clicking its ▶️. If Colab warns *"This notebook was not authored by Google"*, click **Run anyway**.
3. **Wait for ✅ Ready.** It takes 1–2 minutes. The box downloads two small AI models:
   - one that turns text into numbers (section 2);
   - one that writes answers (section 5).

If you see ⚠️ **without a GPU**, everything still works, just more slowly (answers take up to a minute). Raise your hand if you get stuck. An assistant will come and help.

In [ ]:
#@title ▶️ Setup — run me first (takes 1–2 minutes) { display-mode: "form" }
# You don't need to read this cell. It installs what is missing, defines everything the
# workshop does, and loads the two AI models. Curious? All the code is below.
try:
    import sentence_transformers, plotly, ipywidgets  # noqa: F401
except ImportError:
    %pip install -q sentence-transformers transformers accelerate ipywidgets plotly

from dataclasses import dataclass, field
import os
import re
from collections.abc import Iterator
from dataclasses import dataclass
from threading import Thread
from typing import Literal
import numpy as np
import html
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display


# ======================================================================
# content.py
# ======================================================================
"""Every piece of text the workshop uses: the fictional report, the short real passages,
the example sentences and the preset questions."""



@dataclass
class QuestionSpec:
    question: str
    keywords: list[str] = field(default_factory=list)  # a passage "contains the answer" if it has all of them


# ---------------------------------------------------------------- the fictional report
REPORT_TITLE = "Millbrook River Watch — 2025 Annual Report"

REPORT = """\
Millbrook River Watch is a fictional volunteer group, invented for this workshop. None of the people, places or numbers below are real.

A word from the coordinator. This was our eleventh year of looking after the Millbrook River, and our busiest one yet. When Dana Whitlock started the group in 2014, it was six neighbours with a bucket, a notebook and a borrowed thermometer. Today we are a registered charity with a small office above the hardware store on Elm Street, and we share what we learn with the town council every season. I want to thank every volunteer who stood in cold water at seven in the morning so that the rest of the town could know whether the river was healthy.

Our volunteers. In 2025, 143 people volunteered with us at least once, and 38 of them were new this year. Most volunteers join a monthly testing shift, which takes about three hours. A smaller group of twenty people helps with school visits, the newsletter and our spring clean-up. Our youngest volunteer this year was nine years old and came with her grandfather; our oldest was eighty-four. We also welcomed our first group of students from Millbrook High School, who now test two sites as part of their science class.

Where and how we test. We test the water at twelve sites along the river, from the springs above Ashford Farm down to the old mill at the edge of town. At each site, volunteers measure the water temperature, how cloudy the water is, how much oxygen it holds, and the level of E. coli bacteria, which tells us whether sewage or animal waste is getting into the river. Each sample is labelled, kept cold, and carried to our partner laboratory at Harlow College within six hours. The results come back within two days and are entered into our records by a trained volunteer.

What we found. Overall, the river was in slightly better health than in 2024. Oxygen levels were good at ten of the twelve sites for most of the year. Summer was hot and dry, and in August the water at the two lowest sites became warm enough to stress the trout. The site with the highest E. coli readings was Fenwick Bridge, where levels went above the safe swimming limit on nine of our twenty-four visits. We believe the cause is an old storm drain that overflows after heavy rain. We reported this to the town council in September, and repairs to the drain are planned for next summer. Until then, we advise people not to swim or let their dogs swim near Fenwick Bridge for two days after a storm.

Equipment. For years, volunteers carried samples to the laboratory in their own cars, which was slow and hard on people who live far from the river. In June we finally bought a second-hand testing van. It was paid for by the Harlow Valley Community Fund. The van has a small fridge for samples, a fold-out table and space for six volunteers, and it now drives the full testing route twice a month. We also replaced four of our oldest oxygen meters, which had started to give readings that did not match the laboratory results.

Who can see our data. We collect a lot of information, and we take care with it. Our monthly summaries, with one number per site for each measurement, are free for anyone and are posted on the notice board at the public library and in our newsletter. The raw records, which include every single reading, the exact time it was taken and the name of the volunteer who took it, are kept private. Only volunteers who have completed our data training course may open the raw records, and they may not share them outside the group without a vote of the board. We made this rule in 2021 after a volunteer's name and home street were shared online without her permission. Researchers who want the raw records can write to the board, which meets every two months to review such requests.

Money. Our total income in 2025 was a little over forty thousand dollars. About half came from grants, a quarter came from small donations, and the rest came from our spring plant sale and the riverside fun run. Our largest costs were laboratory fees, the van, insurance and printing. We have no paid staff; every hour of work is given freely. A full statement of accounts is available on request from our treasurer, Priya Natarajan.

Working with others. We work closely with the Ashford Farm family, who have planted a strip of trees and shrubs along their stretch of the river to slow down muddy runoff from the fields. Since the trees were planted, the water below the farm has been noticeably clearer after rain. We also lent our equipment to two neighbouring towns that are starting their own river groups, and we ran a training day for their first volunteers in October.

Looking ahead. In 2026 we want to add three new testing sites on the small streams that feed the river, start testing for tiny pieces of plastic, and train at least twenty more volunteers to use the new van. We will also keep pressing the town council until the storm drain at Fenwick Bridge is fixed.

Our annual meeting. Everyone is welcome at our annual meeting, which will be held on March 14, 2026, at 7 p.m. in the community room of the Millbrook Public Library. We will present this report, elect the board for the coming year and share hot soup made by our volunteers. Thank you for caring about our river.
"""

# The preset questions in the chunking section. Each one's answer sits somewhere specific in the report.
VAN_QUESTION = QuestionSpec(
    "Who paid for the group's testing van?",
    ["testing van", "Harlow Valley Community Fund"],
)
RECORDS_QUESTION = QuestionSpec(
    "Who is allowed to look at the raw records?",
    ["raw records", "data training course"],
)
POLLUTION_QUESTION = QuestionSpec(
    "Which site had the most E. coli bacteria?",
    ["Fenwick Bridge", "E. coli"],
)
MEETING_QUESTION = QuestionSpec(
    "When and where is the annual meeting?",
    ["March 14, 2026", "Millbrook Public Library"],
)

REPORT_QUESTIONS = [VAN_QUESTION, RECORDS_QUESTION, POLLUTION_QUESTION, MEETING_QUESTION]

# The answer to this one is spread over two sentences: tiny chunks cut it in half.
SPLIT_QUESTION = VAN_QUESTION

# Used in section 1 (the model alone) and section 5 (the full system). Naming the group makes the
# bare model invent an answer, and it also pulls search toward the report's opening: with top-k = 1 the
# model only sees the wrong chunk; with top-k = 3 it gets the right one (both verified by tests).
PROBLEM_QUESTION = QuestionSpec(
    "When and where is the Millbrook River Watch annual meeting?",
    ["March 14, 2026", "Millbrook Public Library"],
)

# On topic, but the report does not say: a good test of "I don't know".
OUT_OF_DOC_QUESTION = "How much does it cost to become a member of Millbrook River Watch?"

# Chunk sizes (in words) that the "Try this" exercises refer to. All verified by tests/test_effects.py.
DEMO_TINY_SIZE = 10   # chunks shorter than the answer itself: no overlap can save it
DEMO_SMALL_SIZE = 35  # a chunk boundary falls between the two sentences of the van answer...
DEMO_OVERLAP = 15     # ...and this much overlap puts them back together
DEMO_GOOD_SIZE = 60

# ---------------------------------------------------------------- short real passages (search section)
PASSAGES = [
    "Retrieval-Augmented Generation (RAG) combines a language model with a search step. Instead of relying only on what the model memorized during training, the system first retrieves relevant documents from a collection you control, then asks the model to answer using those documents. This grounds answers in your own sources and reduces made-up information.",
    "An embedding is a list of numbers that represents the meaning of a piece of text. Texts with similar meanings have embeddings that point in similar directions, so a computer can find which stored document is most relevant to a question even when they share no exact words.",
    "Ollama is a tool that runs large language models directly on your own computer. Because the model runs locally, the text you send it never leaves your machine, and you do not need an internet connection once the model is downloaded.",
    "Cosine similarity measures how close two embeddings are by looking at the angle between them. A value near 1 means the two texts are very similar in meaning, and a value near 0 means they are unrelated. RAG uses it to rank stored documents against a question.",
    "Running AI models locally matters for data sovereignty. When a community runs its own models on its own hardware, it keeps control over its data and decides who can access it, rather than sending sensitive material to servers owned by outside companies.",
    "Te Hiku Media, a Maori organization in New Zealand, built a speech recognition system for the Maori language. They release their data under a Kaitiakitanga License, which treats data as cared for under guardianship rather than owned, and ensures any benefit derived from the data flows back to the people it came from.",
    "A vector store keeps the embeddings of all your documents so they can be searched quickly. In a small project, a simple list in memory is enough. For larger collections, specialised tools can store and search millions of embeddings in a fraction of a second.",
    "Chunking means splitting long documents into smaller passages before embedding them. Smaller chunks make retrieval more precise, because the system can return just the relevant passage instead of an entire document. Overlapping chunks prevent key sentences from being lost at boundaries.",
]

SEARCH_EXAMPLE = "Who benefits when Maori speech data is shared?"

# ---------------------------------------------------------------- embedding exercises
SIMILARITY_PAIRS = [
    (  # same meaning, almost no shared words
        "Communities have the right to control data collected about them.",
        "Indigenous peoples should govern how their information is stored and used.",
    ),
    (  # shared word, different meaning
        "The bank raised its interest rates again this year.",
        "We had a picnic on the grassy bank of the river.",
    ),
    (  # unrelated
        "Communities have the right to control data collected about them.",
        "I love to ride my bike on Sunday mornings.",
    ),
]

MAP_SENTENCES = [
    ("The river flooded the fields after the storm.", "rivers and water"),
    ("Fish need cold, clean water to lay their eggs.", "rivers and water"),
    ("The lake froze completely in January.", "rivers and water"),
    ("Rain fell on the mountains and filled the streams.", "rivers and water"),
    ("Simmer the soup slowly with onions and garlic.", "food and cooking"),
    ("Fresh bread is best eaten warm from the oven.", "food and cooking"),
    ("She added a pinch of salt to the sauce.", "food and cooking"),
    ("We roasted vegetables for the family dinner.", "food and cooking"),
    ("Communities should decide who can use their data.", "data and privacy"),
    ("Passwords protect your personal information online.", "data and privacy"),
    ("The app shared users' locations without asking.", "data and privacy"),
    ("Medical records must be kept confidential.", "data and privacy"),
    ("He rode his bicycle up the steep mountain pass.", "sport and exercise"),
    ("The team trained every morning before the match.", "sport and exercise"),
    ("She ran her first marathon in under four hours.", "sport and exercise"),
    ("Swimming laps is a gentle way to stay fit.", "sport and exercise"),
]


# ======================================================================
# toolkit.py
# ======================================================================
"""The workshop's machinery: chunking, similarity, search, prompts and the two AI models."""



Strategy = Literal["fixed", "overlap", "sentences", "whole"]


# ---------------------------------------------------------------- chunking
def split_sentences(text: str) -> list[str]:
    """Cut text after each '.', '?' or '!'."""
    return [s.strip() for s in re.split(r"(?<=[.?!])\s+", text) if s.strip()]


def chunk_fixed(text: str, size: int) -> list[str]:
    """Cut text into pieces of `size` words, one after the other."""
    words = text.split()
    return [" ".join(words[i : i + size]) for i in range(0, len(words), size)]


def chunk_overlap(text: str, size: int, overlap: int) -> list[str]:
    """Like chunk_fixed, but each piece repeats the last `overlap` words of the previous one."""
    overlap = max(0, min(overlap, size - 1))
    step = size - overlap
    words = text.split()
    chunks = []
    for i in range(0, len(words), step):
        chunks.append(" ".join(words[i : i + size]))
        if i + size >= len(words):
            break
    return chunks


def chunk_sentences(text: str, n: int) -> list[str]:
    """Group text into pieces of `n` sentences."""
    sentences = split_sentences(text)
    return [" ".join(sentences[i : i + n]) for i in range(0, len(sentences), n)]


def make_chunks(text: str, strategy: Strategy, size: int, overlap: int) -> list[str]:
    """One entry point for every chunking strategy. For 'sentences', size = sentences per chunk."""
    if strategy == "fixed":
        return chunk_fixed(text, size)
    if strategy == "overlap":
        return chunk_overlap(text, size, overlap)
    if strategy == "sentences":
        return chunk_sentences(text, size)
    return [text.strip()]


# ---------------------------------------------------------------- similarity and search
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    """How close two embeddings point: 1 = same direction (same meaning), 0 = unrelated."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def similarity_label(score: float) -> str:
    """Plain-language reading of a similarity score."""
    if score >= 0.60:
        return "very similar"
    if score >= 0.40:
        return "somewhat related"
    if score >= 0.20:
        return "loosely related"
    return "unrelated"


def rank(query_vec: np.ndarray, doc_vecs: np.ndarray, k: int) -> list[tuple[int, float]]:
    """The k stored texts closest to the query, best first, as (position, score)."""
    scores = [cosine(query_vec, d) for d in doc_vecs]
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return [(i, scores[i]) for i in order[: min(k, len(scores))]]


EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"


class MiniLMEmbedder:
    """Turns text into embeddings (384 numbers each) with a small, fast model."""

    def __init__(self, device: str):
        from sentence_transformers import SentenceTransformer

        self.model = SentenceTransformer(EMBED_MODEL, device=device)

    def embed(self, texts: list[str]) -> np.ndarray:
        return self.model.encode(list(texts), normalize_embeddings=True, convert_to_numpy=True)


# ---------------------------------------------------------------- prompt
def build_prompt(question: str, contexts: list[str], allow_unknown: bool) -> str:
    """The exact text sent to the model: instructions, the retrieved passages, the question."""
    lines = ["Answer the question using the context below."]
    if allow_unknown:
        lines.append('If the answer is not in the context, say "I don\'t know based on these documents."')
    lines += ["", "Context:"]
    lines += [f"[{i}] {c}" for i, c in enumerate(contexts, start=1)]
    lines += ["", f"Question: {question}", "Answer:"]
    return "\n".join(lines)


# ---------------------------------------------------------------- map of meaning
def project_2d(vectors: np.ndarray) -> np.ndarray:
    """Squash many-number embeddings down to 2 numbers each (PCA), so they can be drawn on a map."""
    x = np.asarray(vectors, dtype=float)
    x = x - x.mean(axis=0)
    out = np.zeros((len(x), 2))
    if len(x) < 2:
        return out
    _, _, vt = np.linalg.svd(x, full_matrices=False)
    proj = x @ vt[:2].T
    out[:, : proj.shape[1]] = proj
    return out


# ---------------------------------------------------------------- the answer-writing model
CHAT_MODEL_GPU = "Qwen/Qwen3-1.7B"
CHAT_MODEL_CPU = "Qwen/Qwen3-0.6B"
MAX_NEW_TOKENS = 200


class QwenGenerator:
    """Writes answers with a small Qwen model. Greedy decoding: the same question gives the same answer."""

    def __init__(self, model_name: str, device: str):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        dtype = torch.float16 if device == "cuda" else torch.bfloat16
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype).to(device)
        self.model.eval()

    def generate(self, prompt: str) -> Iterator[str]:
        from transformers import TextIteratorStreamer

        text = self.tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        streamer = TextIteratorStreamer(self.tokenizer, skip_prompt=True, skip_special_tokens=True)
        settings = dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            top_k=None,
        )
        worker = Thread(target=self.model.generate, kwargs=settings)
        worker.start()
        yield from streamer
        worker.join()


class FakeGenerator:
    """A stand-in for tests: repeats the first passage instead of thinking."""

    def generate(self, prompt: str) -> Iterator[str]:
        yield "(test answer) "
        first = next((line for line in prompt.splitlines() if line.startswith("[1] ")), prompt)
        for word in first.removeprefix("[1] ").split():
            yield word + " "


@dataclass
class Models:
    embedder: MiniLMEmbedder
    generator: QwenGenerator | FakeGenerator
    device: str
    llm_name: str


def load_models() -> Models:
    """Load both models, on the GPU when there is one."""
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    embedder = MiniLMEmbedder(device)
    if os.environ.get("RAG_WORKSHOP_FAKE_LLM") == "1":
        return Models(embedder, FakeGenerator(), device, "fake")
    name = os.environ.get("RAG_WORKSHOP_LLM") or (CHAT_MODEL_GPU if device == "cuda" else CHAT_MODEL_CPU)
    return Models(embedder, QwenGenerator(name, device), device, name)


def status_line(device: str) -> str:
    if device == "cuda":
        return "✅ Ready — using the GPU. Answers take a few seconds."
    return (
        "⚠️ Ready, but without a GPU: answers will be slow and a bit less accurate. "
        "To fix it: menu Runtime → Change runtime type → T4 GPU → Save, then run this cell again."
    )


def answer(models: Models, question: str, contexts: list[str], allow_unknown: bool) -> Iterator[str]:
    """The whole RAG step: build the prompt from the retrieved passages and stream the model's answer."""
    return models.generator.generate(build_prompt(question, contexts, allow_unknown))


# ======================================================================
# ui.py
# ======================================================================
"""The interactive panels, one per workshop section. Each show_*() displays its panel and returns it."""




# Mid-tone backgrounds with dark text set explicitly, so they read on light and dark themes.
CHUNK_COLOURS = ["#cfe3ff", "#ffe0b3"]
OVERLAP_COLOUR = "#c8efc4"
TEXT = "color:#111"
WIDE = widgets.Layout(width="100%")


def _panel(children: list, controls: dict) -> widgets.VBox:
    box = widgets.VBox(children, layout=widgets.Layout(width="100%", max_width="900px"))
    box.controls = controls
    display(box)
    return box


def _note(message: str) -> str:
    return f'<div style="padding:8px 12px;border-radius:6px;background:#fff3bf;{TEXT}">{html.escape(message)}</div>'


def _bar(score: float, label: str = "") -> str:
    pct = max(0, min(100, round(score * 100)))
    return (
        '<div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap">'
        f'<div style="width:160px;height:14px;background:#e3e3e3;border-radius:7px;overflow:hidden">'
        f'<div style="width:{pct}%;height:100%;background:#3b82f6"></div></div>'
        f"<b>{score:.2f}</b> {html.escape(label)}</div>"
    )


def _box(inner: str, background: str = "#f4f4f5") -> str:
    return f'<div style="padding:10px 12px;border-radius:6px;background:{background};{TEXT};white-space:pre-wrap">{inner}</div>'


def _stream_into(target: widgets.HTML, pieces, button: widgets.Button | None = None, background: str = "#eef6ee") -> None:
    """Write the model's answer into `target` as it arrives. The button stays disabled meanwhile."""
    if button is not None:
        button.disabled = True
    text = ""
    target.value = _box("✍️ The model is writing…", background)
    try:
        for piece in pieces:
            text += piece
            target.value = _box(html.escape(text), background)
    finally:
        if button is not None:
            button.disabled = False
    if not text.strip():
        target.value = _box("(The model gave an empty answer. Try asking again.)", background)


def retrieve_report(m: Models, question: str, k: int) -> list[str]:
    """The k chunks of the report closest to the question (chunks of DEMO_GOOD_SIZE words)."""
    chunks = make_chunks(REPORT, "fixed", DEMO_GOOD_SIZE, 0)
    hits = rank(m.embedder.embed([question])[0], m.embedder.embed(chunks), k)
    return [chunks[i] for i, _ in hits]


# ---------------------------------------------------------------- 1. the problem
def show_problem(m: Models) -> widgets.VBox:
    question = widgets.HTML(_box(f"<b>Question:</b> {html.escape(PROBLEM_QUESTION.question)}"))
    ask = widgets.Button(description="Ask the model", button_style="primary", icon="comment")
    out = widgets.HTML()
    ask.on_click(lambda _: _stream_into(out, m.generator.generate(PROBLEM_QUESTION.question), ask))
    return _panel([question, ask, out], {"ask": ask, "answer": out})


# ---------------------------------------------------------------- 2a. similarity
def show_similarity(m: Models) -> widgets.VBox:
    labels = ["Same meaning, different words", "Same word, different meaning", "Unrelated"]
    pair = widgets.Dropdown(options=[(lab, i) for i, lab in enumerate(labels)], value=0, description="Example:")
    a = widgets.Textarea(value=SIMILARITY_PAIRS[0][0], description="Sentence A", layout=WIDE)
    b = widgets.Textarea(value=SIMILARITY_PAIRS[0][1], description="Sentence B", layout=WIDE)
    compare = widgets.Button(description="Compare", button_style="primary", icon="balance-scale")
    result = widgets.HTML()

    def on_pair(change):
        a.value, b.value = SIMILARITY_PAIRS[change["new"]]
        on_compare(None)

    def on_compare(_):
        if not a.value.strip() or not b.value.strip():
            result.value = _note("Type a sentence first.")
            return
        va, vb = m.embedder.embed([a.value, b.value])
        score = cosine(va, vb)
        result.value = f"Score {_bar(score, '→ ' + similarity_label(score))}"

    pair.observe(on_pair, names="value")
    compare.on_click(on_compare)
    on_compare(None)
    return _panel([pair, a, b, compare, result], {"pair": pair, "a": a, "b": b, "compare": compare, "result": result})


# ---------------------------------------------------------------- 2b. map of meaning
def show_meaning_map(m: Models) -> widgets.VBox:
    base = list(MAP_SENTENCES)
    mine: list[str] = []
    sentence = widgets.Text(placeholder="Type any sentence, then click Add", layout=WIDE)
    add = widgets.Button(description="Add to map", button_style="primary", icon="plus")
    reset = widgets.Button(description="Reset", icon="undo")
    message = widgets.HTML()
    plot = widgets.Output()
    box_ref = {}

    def redraw():
        texts = [s for s, _ in base] + mine
        points = project_2d(m.embedder.embed(texts))
        fig = go.Figure()
        topics = list(dict.fromkeys(t for _, t in base))
        for topic in topics:
            idx = [i for i, (_, t) in enumerate(base) if t == topic]
            fig.add_scatter(
                x=points[idx, 0], y=points[idx, 1], mode="markers", name=topic,
                text=[base[i][0] for i in idx], hoverinfo="text", marker=dict(size=11),
            )
        n = len(base)
        fig.add_scatter(
            x=points[n:, 0], y=points[n:, 1], mode="markers+text", name="your sentences",
            text=[s[:30] + ("…" if len(s) > 30 else "") for s in mine], textposition="top center",
            hovertext=mine, hoverinfo="text", marker=dict(size=15, symbol="star", color="#111"),
        )
        fig.update_layout(
            height=480, margin=dict(l=10, r=10, t=30, b=10), legend=dict(orientation="h", y=-0.05),
            xaxis=dict(showticklabels=False, title=""), yaxis=dict(showticklabels=False, title=""),
            title="Each dot is a sentence. Close dots = close meanings.",
        )
        if "box" in box_ref:
            box_ref["box"].figure = fig
        plot.clear_output(wait=True)
        with plot:
            fig.show()
        return fig

    def on_add(_):
        if not sentence.value.strip():
            message.value = _note("Type a sentence first.")
            return
        mine.append(sentence.value.strip())
        sentence.value = ""
        message.value = ""
        redraw()

    def on_reset(_):
        mine.clear()
        message.value = ""
        redraw()

    add.on_click(on_add)
    reset.on_click(on_reset)
    fig = redraw()
    box = _panel([widgets.HBox([add, reset]), sentence, message, plot],
                 {"sentence": sentence, "add": add, "reset": reset, "message": message})
    box.figure = fig
    box_ref["box"] = box
    return box


# ---------------------------------------------------------------- 3. search
def show_search(m: Models) -> widgets.VBox:
    question = widgets.Text(value=SEARCH_EXAMPLE, description="Question", layout=WIDE)
    search = widgets.Button(description="Search", button_style="primary", icon="search")
    results = widgets.HTML()
    passage_vecs = m.embedder.embed(PASSAGES)

    def on_search(_):
        if not question.value.strip():
            results.value = _note("Type a question first.")
            return
        hits = rank(m.embedder.embed([question.value])[0], passage_vecs, len(PASSAGES))
        rows = []
        for place, (i, score) in enumerate(hits, start=1):
            bg = "#dbeafe" if place <= 3 else "#f4f4f5"
            rows.append(
                f'<div class="rag-row" style="padding:8px 10px;margin:4px 0;border-radius:6px;background:{bg};{TEXT}">'
                f"<b>#{place}</b> {_bar(score, '→ ' + similarity_label(score))}"
                f'<div style="margin-top:4px">{html.escape(PASSAGES[i])}</div></div>'
            )
        results.value = "<div><i>Blue = the 3 passages a RAG system would hand to the model.</i></div>" + "".join(rows)

    search.on_click(on_search)
    on_search(None)
    return _panel([question, search, results], {"question": question, "search": search, "results": results})


# ---------------------------------------------------------------- 4. chunking
def _render_chunks(words: list[str], strategy: str, chunks: list[str], size: int, overlap: int, winner: int) -> str:
    """The report with each chunk in alternating colours; words shared by two chunks in green."""
    owners: list[list[int]] = [[] for _ in words]
    if strategy == "overlap":
        step = size - max(0, min(overlap, size - 1))
        for ci in range(len(chunks)):
            for w in range(ci * step, min(ci * step + size, len(words))):
                owners[w].append(ci)
    else:
        pos = 0
        for ci, chunk in enumerate(chunks):
            for w in range(pos, min(pos + len(chunk.split()), len(words))):
                owners[w].append(ci)
            pos += len(chunk.split())
    parts = []
    for word, own in zip(words, owners):
        colour = OVERLAP_COLOUR if len(own) > 1 else CHUNK_COLOURS[own[0] % 2] if own else "transparent"
        style = f"background:{colour};{TEXT}"
        if winner in own:
            style += ";font-weight:bold;text-decoration:underline"
        parts.append(f'<span style="{style}">{html.escape(word)}</span>')
    return (
        f'<div style="max-height:320px;overflow-y:auto;padding:10px;border:1px solid #ccc;border-radius:6px;'
        f'background:#fff;{TEXT};line-height:1.7">' + " ".join(parts) + "</div>"
    )


def show_chunking(m: Models) -> widgets.VBox:
    strategy = widgets.Dropdown(
        options=[("Fixed size", "fixed"), ("Fixed size with overlap", "overlap"),
                 ("By sentences", "sentences"), ("Whole document — no chunking", "whole")],
        value="fixed", description="How to cut",
    )
    size = widgets.IntSlider(value=DEMO_GOOD_SIZE, min=10, max=200, step=5, description="Words/chunk",
                             continuous_update=False)
    overlap = widgets.IntSlider(value=DEMO_OVERLAP, min=0, max=50, step=5, description="Overlap",
                                disabled=True, continuous_update=False)
    question = widgets.Dropdown(options=[q.question for q in REPORT_QUESTIONS], description="Question",
                                layout=widgets.Layout(width="100%", max_width="560px"))
    count = widgets.HTML()
    text = widgets.HTML()
    top = widgets.HTML()
    ask = widgets.Button(description="Ask the model with this chunk", button_style="primary", icon="comment",
                         layout=widgets.Layout(width="auto"))
    out = widgets.HTML()
    words = REPORT.split()
    cache: dict = {}
    state = {"switching": False, "top": ""}

    def current_chunks():
        key = (strategy.value, size.value, overlap.value if strategy.value == "overlap" else 0)
        if key not in cache:
            chunks = make_chunks(REPORT, strategy.value, size.value, key[2])
            cache[key] = (chunks, m.embedder.embed(chunks))
        return cache[key]

    def update(_=None):
        if state["switching"]:
            return
        overlap.disabled = strategy.value != "overlap"
        size.disabled = strategy.value == "whole"
        chunks, vecs = current_chunks()
        (i, score), = rank(m.embedder.embed([question.value])[0], vecs, 1)
        state["top"] = chunks[i]
        count.value = f"<b>{len(chunks)} chunk{'s' if len(chunks) != 1 else ''}</b> — {REPORT_TITLE}"
        text.value = _render_chunks(words, strategy.value, chunks, size.value, overlap.value, i)
        top.value = (
            f"<b>Chunk found for this question</b> (chunk #{i + 1}) {_bar(score)}"
            + _box(html.escape(chunks[i]), "#dbeafe")
        )
        out.value = ""

    def on_strategy(change):
        state["switching"] = True
        if change["new"] == "sentences":
            size.min, size.step, size.value, size.max = 1, 1, 3, 10
            size.description = "Sentences"
        elif change["old"] == "sentences":
            size.max, size.value, size.min, size.step = 200, DEMO_GOOD_SIZE, 10, 5
            size.description = "Words/chunk"
        state["switching"] = False
        update()

    def on_ask(_):
        contexts = [state["top"]]
        _stream_into(out, answer(m, question.value, contexts, allow_unknown=True), ask)

    strategy.observe(on_strategy, names="value")
    for w in (size, overlap, question):
        w.observe(update, names="value")
    ask.on_click(on_ask)
    update()
    controls = {"strategy": strategy, "size": size, "overlap": overlap, "question": question,
                "count": count, "text": text, "top": top, "ask": ask, "answer": out}
    return _panel([widgets.HBox([strategy, size, overlap], layout=widgets.Layout(flex_flow="row wrap")),
                   question, count, text, top, ask, out], controls)


# ---------------------------------------------------------------- 5. generation
def show_generation(m: Models) -> widgets.VBox:
    question = widgets.Text(value=PROBLEM_QUESTION.question, description="Question", layout=WIDE)
    topk = widgets.IntSlider(value=3, min=1, max=5, description="Passages (k)")
    unknown = widgets.Checkbox(value=True, description="Allow 'I don't know'", indent=False)
    ask = widgets.Button(description="Ask", button_style="primary", icon="comment")
    prompt = widgets.HTML()
    rag = widgets.HTML()
    norag = widgets.HTML()
    column = widgets.Layout(flex="1 1 300px", min_width="260px", margin="4px")
    columns = widgets.Box(
        [widgets.VBox([widgets.HTML("<b>With RAG</b> (reads the passages)"), rag], layout=column),
         widgets.VBox([widgets.HTML("<b>Without RAG</b> (memory only)"), norag], layout=column)],
        layout=widgets.Layout(display="flex", flex_flow="row wrap", width="100%"),
    )
    controls = {"question": question, "topk": topk, "unknown": unknown, "ask": ask,
                "prompt": prompt, "rag": rag, "norag": norag, "prompt_text": ""}

    def on_ask(_):
        q = question.value.strip()
        if not q:
            prompt.value = _note("Type a question first.")
            return
        ask.disabled = True
        try:
            contexts = retrieve_report(m, q, topk.value)
            text = build_prompt(q, contexts, unknown.value)
            controls["prompt_text"] = text
            shown = html.escape(text)
            for ctx in contexts:
                esc = html.escape(ctx)
                shown = shown.replace(esc, f'<mark style="background:#dbeafe;{TEXT}">{esc}</mark>', 1)
            prompt.value = "<b>This is exactly what the model receives:</b>" + _box(shown)
            rag.value = norag.value = ""
            _stream_into(rag, m.generator.generate(text))
            _stream_into(norag, m.generator.generate(q), background="#fdecec")
        finally:
            ask.disabled = False

    ask.on_click(on_ask)
    return _panel([question, widgets.HBox([topk, unknown], layout=widgets.Layout(flex_flow="row wrap")),
                   ask, prompt, columns], controls)


# ---------------------------------------------------------------- bonus: your own documents
def load_uploads(files: dict[str, bytes]) -> tuple[list[str], list[str]]:
    """Turn uploaded files into chunks. Returns (chunks, messages about skipped files)."""
    chunks, messages = [], []
    for name, data in files.items():
        if not name.lower().endswith((".txt", ".md")):
            messages.append(f"⚠️ {name} is not a .txt file — skipped. Save it as plain text (.txt) and try again.")
            continue
        try:
            text = bytes(data).decode("utf-8")
        except UnicodeDecodeError:
            messages.append(f"⚠️ {name} could not be read as text — skipped. Save it as UTF-8 plain text and try again.")
            continue
        if not text.strip():
            messages.append(f"⚠️ {name} is empty — skipped.")
            continue
        chunks += make_chunks(text, "overlap", DEMO_GOOD_SIZE, DEMO_OVERLAP)
    return chunks, messages


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


def show_upload(m: Models) -> widgets.VBox:
    status = widgets.HTML()
    question = widgets.Text(placeholder="Ask a question about your files", description="Question", layout=WIDE)
    ask = widgets.Button(description="Ask", button_style="primary", icon="comment")
    out = widgets.HTML()
    store = {"chunks": [], "vecs": None}

    def take(files: dict[str, bytes]):
        chunks, messages = load_uploads(files)
        store["chunks"] = chunks
        store["vecs"] = m.embedder.embed(chunks) if chunks else None
        lines = [html.escape(msg) for msg in messages]
        lines.append(f"✅ {len(chunks)} chunks ready." if chunks else "No usable text yet.")
        status.value = _box("<br>".join(lines))

    if _in_colab():
        picker = widgets.Button(description="Choose .txt files", icon="upload")

        def on_pick(_):
            from google.colab import files as colab_files

            take(colab_files.upload())

        picker.on_click(on_pick)
    else:
        picker = widgets.FileUpload(accept=".txt,.md", multiple=True, description="Choose .txt files")
        picker.observe(lambda ch: take({f["name"]: f["content"] for f in ch["new"]}), names="value")

    def on_ask(_):
        if not store["chunks"]:
            out.value = _note("Upload at least one .txt file first.")
            return
        if not question.value.strip():
            out.value = _note("Type a question first.")
            return
        hits = rank(m.embedder.embed([question.value])[0], store["vecs"], 3)
        contexts = [store["chunks"][i] for i, _ in hits]
        _stream_into(out, answer(m, question.value, contexts, allow_unknown=True), ask)

    ask.on_click(on_ask)
    return _panel([picker, status, question, ask, out],
                  {"picker": picker, "status": status, "question": question, "ask": ask, "answer": out})


print("Loading the two AI models (the first time takes 1–2 minutes)…")
MODELS = load_models()
print(status_line(MODELS.device))


## 1. The problem: an AI that guesses ⏱ about 5 min

A language model (the kind of AI behind chatbots) has read a huge amount of text, and it answers from what it remembers. But it knows nothing about documents it never saw: your organisation's reports, a community's records, or anything written after it was trained. When it doesn't know, it often doesn't say so. Instead it writes something that *sounds* right. This is called a **hallucination**.

Throughout this workshop we will use the annual report of **Millbrook River Watch**, a volunteer group that tests the water of a town's river. The group and its report are **fictional**: we invented them so that the AI cannot possibly know them already. You will see the full report in section 4.

> **What you'll see:** the model answers a question about Millbrook River Watch's annual meeting without having seen the report.
>
> **Try this**
> 1. Click ▶️ below, then click **Ask the model**.
> 2. Read the answer. Does it sound sure of itself?
> 3. Compare with the truth from the report: the meeting is on **March 14, 2026, at the Millbrook Public Library**.
>
> **What to notice:** the model does not say "I don't know". It guesses, and the guess sounds convincing.

In [ ]:
#@title ▶️ Ask the model, without the report { display-mode: "form" }
if "MODELS" not in globals():
    print("⚠️ Please run the Setup cell (section 0) first, then run this cell again.")
else:
    show_problem(MODELS)


**What if, before answering, we could hand the model the right page of the report?** That is exactly what RAG does:

```
 your question ──► 🔎 search your documents ──► 📄 the best passages ──► 🤖 the model writes the answer
```

To search by meaning, the computer first needs a way to tell whether two pieces of text mean similar things. That is the job of **embeddings**.

## 2. Embeddings: turning meaning into numbers ⏱ about 20 min

Computers can't compare meanings directly, but they are very good at comparing numbers. An **embedding model** reads a piece of text and turns it into a list of numbers (here, 384 of them). Think of these numbers as the **coordinates of the sentence on a giant map of meaning**. Sentences that mean similar things land close together, even when they use completely different words.

To measure how close two sentences are, we use a **similarity score**. A score near **1** means the two sentences are about the same thing, and a score near **0** means they have nothing in common. With this model, a score around 0.5 already means "close".

### 2a. Compare two sentences

> **What you'll see:** two sentences and their similarity score.
>
> **Try this**
> 1. Look at the first example, *"Same meaning, different words"*. The two sentences share almost no words. What score do they get?
> 2. Pick *"Same word, different meaning"* in the **Example** menu. Both sentences contain the word "bank". Is the score high?
> 3. Write your own pair, then click **Compare**. Try a sentence and its opposite, like *"I love winter."* and *"I hate winter."*
>
> **What to notice:** the score measures whether two sentences are **about the same thing**, not whether they agree. Opposites are often "close" because they talk about the same topic.

In [ ]:
#@title ▶️ Compare two sentences { display-mode: "form" }
if "MODELS" not in globals():
    print("⚠️ Please run the Setup cell (section 0) first, then run this cell again.")
else:
    show_similarity(MODELS)


### 2b. The map of meaning

384 numbers can't be drawn on a screen, so we squash them down to just 2. It's a bit like the shadow of a 3D object on a wall: you keep the general shape but lose some detail. On the map below, each dot is a sentence, and the colours show four topics we picked.

> **What you'll see:** 16 sentences on a map, grouped by meaning.
>
> **Try this**
> 1. Hover over the dots to read the sentences. Do the colours form groups?
> 2. Type *"My grandmother bakes the best apple pie."* and click **Add to map**. Where does your ★ land?
> 3. Add a sentence that mixes two topics, like *"We cooked the fish we caught in the river."* Then add one in another language, like *"La rivière déborde après la pluie."* (this model learned mostly from English text).
>
> **What to notice:** nobody told the computer what the topics are. The groups appear on their own, from the meaning alone. The map moves a little each time you add a sentence, because squashing 384 numbers into 2 is only an approximation.

In [ ]:
#@title ▶️ The map of meaning { display-mode: "form" }
if "MODELS" not in globals():
    print("⚠️ Please run the Setup cell (section 0) first, then run this cell again.")
else:
    show_meaning_map(MODELS)


<details><summary>💡 For the curious</summary>

The embedding model is `all-MiniLM-L6-v2`. It is small (about 90 MB) and fast, and it was trained on more than a billion pairs of related sentences. The similarity score is called **cosine similarity**: it measures the angle between two lists of numbers seen as arrows. The squashing to 2D is called **PCA** (principal component analysis).

</details>

## 3. Search: finding the right passage ⏱ about 10 min

Now that we can measure how close two texts are, searching is simple:

1. **In advance**, turn every passage of your collection into an embedding.
2. **When a question arrives**, turn it into an embedding too.
3. **Rank** the passages by their similarity to the question. The top few (here, 3) are what the AI will read.

Below is a small collection of 8 real passages about AI and data sovereignty. One of them is about Te Hiku Media, a Māori organisation that shares its language data under a *Kaitiakitanga License*.

> **What you'll see:** all 8 passages, ranked from most to least relevant to your question.
>
> **Try this**
> 1. Click **Search** with the question already there. Which passage comes first? Does it contain the words of your question?
> 2. Ask *"Can I use AI without the internet?"* Which passage wins, and why?
> 3. Ask *"What's the best pizza topping?"* None of the passages is about pizza. Look at the scores.
>
> **What to notice:** search works with meaning, not exact words, but it **always returns something**, even when nothing is relevant. Remember this for section 5.

In [ ]:
#@title ▶️ Search the passages { display-mode: "form" }
if "MODELS" not in globals():
    print("⚠️ Please run the Setup cell (section 0) first, then run this cell again.")
else:
    show_search(MODELS)


## 4. Chunking: cutting documents into pieces ⏱ about 20 min

Real documents are long. An embedding is always one list of 384 numbers, whether the text is one line or a whole report. Squeezing a 1,000-word report into a single embedding blurs it, like summarising a book in one sentence. So before searching, we cut documents into smaller pieces called **chunks**, and give each chunk its own embedding. Search can then return the precise piece that answers the question.

**How you cut matters:**
- **Chunks too big:** the embedding gets blurry, and the AI receives a lot of text that has nothing to do with the question.
- **Chunks too small:** an answer can be cut in half.
- **A cut in the wrong place:** a fact that spans two sentences ends up split between two chunks.
- **Overlap** is a common fix: each chunk repeats the last few words of the one before, like roof tiles that overlap so no rain gets through.

In the panel below, the Millbrook River Watch report is coloured by chunk: **blue and orange** take turns, **green** words belong to two chunks (overlap), and the chunk that search found is in **bold and underlined**.

> **What you'll see:** the report cut into chunks, the chunk found for your question, and the model's answer when it reads only that chunk.
>
> **Try this**
> 1. Keep *Fixed size, 60 words* and the question *"Who paid for the group's testing van?"*. Click **Ask the model with this chunk**. It finds the answer.
> 2. Choose **Whole document — no chunking**. The score of the match drops: the whole report is "about" too many things at once.
> 3. Back to **Fixed size**, set **10 words**. The chunk found now holds only part of the answer. Ask the model.
> 4. Set **35 words**. The size looks reasonable, but the cut falls right between *"…a second-hand testing van."* and *"It was paid for by the Harlow Valley Community Fund."* Ask the model.
> 5. Switch to **Fixed size with overlap**, keep 35 words, and set the overlap to **15**. The answer is back in one piece. Look for the green words.
> 6. Explore the other questions and the **By sentences** option.
>
> **What to notice:** no chunk size is perfect for every question. Real systems try several, and often cut at natural boundaries like paragraphs, with some overlap.

In [ ]:
#@title ▶️ Cut the report into chunks { display-mode: "form" }
if "MODELS" not in globals():
    print("⚠️ Please run the Setup cell (section 0) first, then run this cell again.")
else:
    show_chunking(MODELS)


## 5. Generation: writing the answer ⏱ about 15 min

The last step puts everything together. The system takes the question, finds the **k** closest chunks of the report (k is just "how many"), and sends the model one piece of text: **instructions + those chunks + the question**. There is nothing magic about it. The model reads that text, the way you would, and writes an answer. The panel shows you exactly what the model receives, with the chunks highlighted in blue.

> **What you'll see:** the exact text sent to the model, then two answers side by side: one with RAG (it read the chunks) and one without (memory only).
>
> **Try this**
> 1. Click **Ask** with the question about the annual meeting and 3 passages. Compare the two columns.
> 2. Set **Passages (k)** to **1** and ask again. The single closest chunk is now the report's introduction (it mentions the group's name), not the meeting. What does the model say?
> 3. Ask *"How much does it cost to become a member of Millbrook River Watch?"* The report doesn't say. Try it with **Allow 'I don't know'** ticked, then unticked.
> 4. Ask your own questions about the report (you can read it in section 4).
>
> **What to notice:** the model can only be as good as the chunks it is given. When search hands it the wrong chunk, it cannot answer, and it may guess. Most RAG mistakes are really search mistakes.

In [ ]:
#@title ▶️ Ask with and without RAG { display-mode: "form" }
if "MODELS" not in globals():
    print("⚠️ Please run the Setup cell (section 0) first, then run this cell again.")
else:
    show_generation(MODELS)


## 6. Take it home ⏱ about 10 min

### What you built, in one picture

```
 PREPARE (once)   your documents ──► ✂️ chunking ──► 🔢 embeddings ──► stored

 ASK (each time)  question ──► 🔢 embedding ──► 🔎 search: closest chunks
                            ──► 📝 prompt = instructions + chunks + question ──► 🤖 answer
```

Every choice changes the answers you get: how you cut, which embedding model, how many chunks you hand over (k), and what the instructions say.

### Running it on your own computer

In Colab, your text is processed on Google's computers. On your own computer, your documents **never leave it**. For sensitive material, that difference matters. There are two ways to run it yourself:

1. **This same notebook.** Install Python (version 3.11 or newer), then in a terminal run
   `pip install sentence-transformers transformers torch accelerate ipywidgets plotly jupyterlab`.
   Download this notebook (**File → Download → Download .ipynb**), start `jupyter lab`, open the notebook and run it the same way. The models download once (about 1–2 GB) and then work without internet. On a computer without a GPU it uses the smaller answer model automatically.
2. **Ollama**, a free app that runs AI models on your computer. Our companion notebook, [`ollama_rag_workshop.ipynb`](https://github.com/hugobb/IndigenousAI/blob/main/notebooks/ollama_rag_workshop.ipynb), builds the same system with Ollama and shows all the code step by step.

### What should go in?

Keeping data on your own machine does not by itself answer the question of whether something *should* be digitised or put into an AI system at all. Those are community decisions about protocols and access, and they come before the technology, not after it.

The **OCAP® principles** (Ownership, Control, Access, Possession), developed by the First Nations Information Governance Centre, state that First Nations communities collectively own and control their data. Even our fictional river group had to decide who may see its raw records, and wrote a rule after someone's name was shared without permission.

**To discuss**
1. Who should decide what goes into a RAG system about a community, and who may ask it questions?
2. The model can still be wrong, even with RAG. Who is responsible when an answer is wrong?
3. What, in your own work, would you put in such a system, and what would you never put in?

## ★ Bonus: try your own documents

Upload one or more **plain-text files** (`.txt`), then ask questions about them. Word or Google Docs files can be saved as plain text: **File → Download → Plain text (.txt)**. Your files are cut into chunks of 60 words with an overlap of 15, and the model reads the 3 closest chunks.

⚠️ **In Colab, uploaded files go to Google's computers.** Only upload text you are comfortable sharing, and that is yours to share. For anything sensitive, run the notebook on your own computer (see section 6).

> **Try this**
> 1. Upload a document you know well, and ask a question you know the answer to. Is the answer right?
> 2. Ask about something that is *not* in your document. Does the model say it doesn't know?

In [ ]:
#@title ▶️ Your own documents { display-mode: "form" }
if "MODELS" not in globals():
    print("⚠️ Please run the Setup cell (section 0) first, then run this cell again.")
else:
    show_upload(MODELS)
